In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

DATA_PATH = Path("../data")

In [ ]:
# -------------------------
# 1) Chargement des données
# -------------------------
train_gdf = gpd.read_file(DATA_PATH / "train.geojson", index_col=0)
test_gdf  = gpd.read_file(DATA_PATH / "test.geojson", index_col=0)

/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option INDEX_COL
  return ogr_read(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option INDEX_COL
  return ogr_read(


In [ ]:
# Projection en mètres pour avoir des surfaces et périmètres exploitables
train_gdf = train_gdf.to_crs(epsg=3857)
test_gdf = test_gdf.to_crs(epsg=3857)

In [ ]:
# -------------------------
# Vérification des dimensions
# -------------------------
print("Train dataset : ", train_gdf.shape, "->", train_gdf.shape[0], "lignes,", train_gdf.shape[1], "colonnes")
print("Test dataset  : ", test_gdf.shape, "->", test_gdf.shape[0], "lignes,", test_gdf.shape[1], "colonnes")

Train dataset :  (296146, 45) -> 296146 lignes, 45 colonnes
Test dataset  :  (120526, 44) -> 120526 lignes, 44 colonnes


In [ ]:
def safe_convex_hull_area(geom):
    """Calcule la surface de l'enveloppe convexe sans planter sur les géométries corrompues."""
    if geom is None or geom.is_empty:
        return 0.0
    try:
        return geom.convex_hull.area
    except Exception:
        # Capture les GEOSException (NaN/Inf)
        return 0.0

def safe_num_vertices(geom):
    """Compte les sommets, compatible avec les Polygons ET les MultiPolygons."""
    if geom is None or geom.is_empty:
        return 0
    try:
        if geom.geom_type == 'Polygon':
            return len(geom.exterior.coords)
        elif geom.geom_type == 'MultiPolygon':
            return sum(len(poly.exterior.coords) for poly in geom.geoms)
        return 0
    except Exception:
        return 0

def add_geometry_features(gdf):
    # L'utilisation vectorisée de geopandas tolère les NaN/Inf avec un simple warning.
    # On force le remplacement des NaN éventuels par 0.
    gdf["area"] = gdf.geometry.area.fillna(0)
    gdf["perimeter"] = gdf.geometry.length.fillna(0)

    gdf["compactness"] = 4 * np.pi * gdf["area"] / (gdf["perimeter"]**2 + 1e-6)

    bounds = gdf.geometry.bounds
    width  = (bounds["maxx"] - bounds["minx"]).fillna(0)
    height = (bounds["maxy"] - bounds["miny"]).fillna(0)

    gdf["bbox_ratio"] = width / (height + 1e-6)
    gdf["aspect_ratio"] = np.maximum(width, height) / (np.minimum(width, height) + 1e-6)

    # ---  ANTI-CRASH  ---
    gdf["num_vertices"] = gdf.geometry.apply(safe_num_vertices)
    gdf["convex_hull_area"] = gdf.geometry.apply(safe_convex_hull_area)
    # -------------------------------------------

    gdf["log_area"] = np.log1p(gdf["area"])
    gdf["log_perimeter"] = np.log1p(gdf["perimeter"])

    # Shape advanced
    gdf["solidity"] = gdf["area"] / (gdf["convex_hull_area"] + 1e-6)
    gdf["elongation"] = width / (height + 1e-6)
    gdf["rectangularity"] = gdf["area"] / (width * height + 1e-6)

    # nettoyer tout NaN généré par des divisions bizarres
    cols_to_fill = ["compactness", "bbox_ratio", "aspect_ratio", "solidity", "elongation", "rectangularity"]
    gdf[cols_to_fill] = gdf[cols_to_fill].replace([np.inf, -np.inf], 0).fillna(0)

    return gdf

# Relancez l'application :
train_gdf = add_geometry_features(train_gdf)
test_gdf  = add_geometry_features(test_gdf)

/usr/local/lib/python3.12/dist-packages/shapely/measurement.py:50: RuntimeWarning: invalid value encountered in area
  return lib.area(geometry, **kwargs)
/usr/local/lib/python3.12/dist-packages/shapely/measurement.py:196: RuntimeWarning: invalid value encountered in length
  return lib.length(geometry, **kwargs)
/usr/local/lib/python3.12/dist-packages/shapely/measurement.py:50: RuntimeWarning: invalid value encountered in area
  return lib.area(geometry, **kwargs)
/usr/local/lib/python3.12/dist-packages/shapely/measurement.py:196: RuntimeWarning: invalid value encountered in length
  return lib.length(geometry, **kwargs)


In [ ]:
# =========================================================
# 3) Temporal, Status AND RGB Synchronized Sorting
# =========================================================
date_cols = [f"date{i}" for i in range(5)]
status_cols = [f"change_status_date{i}" for i in range(5)]

status_order = {
    "Greenland": 0, "Prior Construction": 1, "Land Cleared": 2,
    "Materials Dumped": 3, "Materials Introduced": 4, "Excavation": 5,
    "Construction Started": 6, "Construction Midway": 7, "Construction Done": 8, "Operational": 9
}

def process_status_features(gdf):
    # 1) Parser les dates
    for c in date_cols:
        gdf[c] = pd.to_datetime(gdf[c], dayfirst=True, errors="coerce")

    # 2) Extraction NumPy (Dates, Statuts et Pixels)
    dates_arr = gdf[date_cols].values.astype("datetime64[ns]")
    status_arr = gdf[status_cols].values

    colors = ["red", "green", "blue"]
    pixel_data = {}
    for color in colors:
        # Attention au décalage : img_red_mean_date1 correspond à date0
        pixel_data[f"{color}_mean"] = gdf[[f"img_{color}_mean_date{i}" for i in range(1, 6)]].values
        pixel_data[f"{color}_std"] = gdf[[f"img_{color}_std_date{i}" for i in range(1, 6)]].values

    # 3) Calcul de l'index chronologique
    sort_idx = np.argsort(dates_arr, axis=1)
    row_idx = np.arange(len(gdf))[:, None]

    # 4) Application du tri simultané
    sorted_dates = dates_arr[row_idx, sort_idx]
    sorted_statuses = status_arr[row_idx, sort_idx]

    # 5) Réinjection de TOUTES les données triées dans le DataFrame
    for i in range(5):
        # Dates et statuts
        gdf[date_cols[i]] = sorted_dates[:, i]
        gdf[status_cols[i]] = sorted_statuses[:, i]
        gdf[f"status_num_date{i}"] = gdf[status_cols[i]].map(status_order)

        # Pixels (Synchronisés avec la bonne date !)
        for color in colors:
            gdf[f"img_{color}_mean_date{i+1}"] = pixel_data[f"{color}_mean"][row_idx, sort_idx][:, i]
            gdf[f"img_{color}_std_date{i+1}"] = pixel_data[f"{color}_std"][row_idx, sort_idx][:, i]

    # --- Calcul des features temporelles sur les données triées ---
    gdf["total_duration_days"] = (gdf["date4"] - gdf["date0"]).dt.days
    gdf["status_progress"] = gdf["status_num_date4"] - gdf["status_num_date0"]
    gdf["construction_speed"] = (gdf["status_progress"] / gdf["total_duration_days"].replace(0, np.nan)) * 365.0

    for i in range(4):
        gdf[f"delta_status_{i}_{i+1}"] = gdf[f"status_num_date{i+1}"] - gdf[f"status_num_date{i}"]
        gdf[f"delta_days_{i}_{i+1}"] = (gdf[f"date{i+1}"] - gdf[f"date{i}"]).dt.days

    gdf["nb_status_changes"] = (gdf[[f"delta_status_{i}_{i+1}" for i in range(4)]] != 0).sum(axis=1)
    gdf["first_status"] = gdf["status_num_date0"]
    gdf["last_status"] = gdf["status_num_date4"]
    gdf["status_range"] = gdf["last_status"] - gdf["first_status"]
    gdf["is_monotonic_increasing"] = (gdf[[f"delta_status_{i}_{i+1}" for i in range(4)]] >= 0).all(axis=1).astype(int)

    return gdf

print("Application du tri chronologique global...")
train_gdf = process_status_features(train_gdf)
test_gdf  = process_status_features(test_gdf)

Application du tri chronologique global...


In [ ]:
# -------------------------
# 4) Urban / Geography Multi-hot (Version Turbo)
# -------------------------
def multi_hot_optimized(df, col):
    # Vérification si la colonne existe pour éviter le KeyError au second run
    if col not in df.columns:
        return df

    # 1. Nettoyage initial
    s = df[col].astype(str)
    # Remplacement ciblé des valeurs nulles
    s = s.replace(["N,A", "NA", "NaN", "nan", "None", "<NA>"], pd.NA)

    # 2. Supprimer les espaces autour des virgules
    s = s.str.replace(r"\s*,\s*", ",", regex=True).str.strip()

    # 3. L'astuce vectorisée : get_dummies gère le split ET le multi-hot d'un coup !
    dummies = s.dropna().str.get_dummies(sep=",")

    # 4. Préfixer les colonnes pour éviter les conflits de noms
    dummies = dummies.add_prefix(f"{col}_")

    # 5. Réintégrer dans le DataFrame
    dummies = dummies.reindex(df.index, fill_value=0)
    df = df.drop(columns=[col]).join(dummies)

    return df

# Application ultra-rapide avec vérification
train_gdf = multi_hot_optimized(train_gdf, "urban_type")
train_gdf = multi_hot_optimized(train_gdf, "geography_type")
test_gdf  = multi_hot_optimized(test_gdf, "urban_type")
test_gdf  = multi_hot_optimized(test_gdf, "geography_type")

# Alignement des colonnes train/test
train_gdf, test_gdf = train_gdf.align(test_gdf, join="left", axis=1, fill_value=0)

In [ ]:
# Affiche toutes les colonnes qui contiennent le mot "red"
print([col for col in train_gdf.columns if 'red' in col.lower()])

['img_red_mean_date1', 'img_red_std_date1', 'img_red_mean_date2', 'img_red_std_date2', 'img_red_mean_date3', 'img_red_std_date3', 'img_red_mean_date4', 'img_red_std_date4', 'img_red_mean_date5', 'img_red_std_date5']


In [ ]:
 # =========================================================
# 5) RGB / Satellite Features (Maintenant sur des bases saines !)
# =========================================================
colors = ["red", "green", "blue"]

def process_rgb(df):
    for color in colors:
        mean_cols = [f"img_{color}_mean_date{i}" for i in range(1, 6)]
        std_cols  = [f"img_{color}_std_date{i}"  for i in range(1, 6)]

        # Moyennes temporelles
        df[f"{color}_mean_temporal"] = df[mean_cols].mean(axis=1)
        df[f"{color}_std_temporal"]  = df[std_cols].mean(axis=1)

        # Pente (Slope) - Maintenant valide car les dates 1, 2, 4, 5 sont dans le bon ordre
        df[f"{color}_slope"] = (
            - 2 * df[f"img_{color}_mean_date1"]
            - 1 * df[f"img_{color}_mean_date2"]
            + 1 * df[f"img_{color}_mean_date4"]
            + 2 * df[f"img_{color}_mean_date5"]
        ) / 10.0

        # Différences successives chronologiques
        for i in range(1, 5):
            df[f"d_{color}_mean_{i}_{i+1}"] = (
                df[f"img_{color}_mean_date{i+1}"] - df[f"img_{color}_mean_date{i}"]
            )
            df[f"d_{color}_std_{i}_{i+1}"] = (
                df[f"img_{color}_std_date{i+1}"] - df[f"img_{color}_std_date{i}"]
            )

        # Variabilité
        df[f"{color}_range_mean"] = df[mean_cols].max(axis=1) - df[mean_cols].min(axis=1)
        df[f"{color}_cv"] = df[std_cols].mean(axis=1) / (df[mean_cols].mean(axis=1) + 1e-6)

    # Ratios sécurisés
    df["green_ratio"] = df["green_mean_temporal"] / (df["red_mean_temporal"] + df["blue_mean_temporal"] + 1e-6)
    df["red_ratio"] = df["red_mean_temporal"] / (df["green_mean_temporal"] + df["blue_mean_temporal"] + 1e-6)

    return df

print("Calcul des signaux RGB chronologiques...")
train_gdf = process_rgb(train_gdf)
test_gdf  = process_rgb(test_gdf)

Calcul des signaux RGB chronologiques...


In [ ]:
# -------------------------
# 6) Interactions potentiels
# -------------------------


# -------------------------------
# Geometry × Status
# -------------------------------
def add_geometry_status_interactions(df):
    # Surface × nombre de changements de status
    df["area_x_nb_status_changes"] = df["area"] * df["nb_status_changes"]
    df["area_x_status_progress"] = df["area"] * df["status_progress"]

    # Compacité × max variation rouge
    red_max_jump = df[[f"d_red_mean_{i}_{i+1}" for i in range(1,5)]].max(axis=1)
    df["compactness_x_max_red_jump"] = df["compactness"] * red_max_jump
    return df

# -------------------------------
# RGB × Geometry
# -------------------------------
def add_rgb_geometry_interactions(df):
    for color in ["red", "green", "blue"]:
        # slope × area
        df[f"{color}_slope_x_area"] = df[f"{color}_slope"] * df["area"]

        # mean difference max × bbox_ratio
        max_d = df[[f"d_{color}_mean_{i}_{i+1}" for i in range(1,5)]].max(axis=1)
        df[f"max_d_{color}_mean_x_bbox_ratio"] = max_d * df["bbox_ratio"]
    return df

# -------------------------------
# RGB × Status
# -------------------------------
def add_rgb_status_interactions(df):
    for color in ["red", "green", "blue"]:
        max_d = df[[f"d_{color}_mean_{i}_{i+1}" for i in range(1,5)]].max(axis=1)
        # Max delta × status_progress
        df[f"{color}_max_delta_x_status_progress"] = max_d * df["status_progress"]
    return df

# -------------------------------
# Triple interactions (Geometry × RGB × Status)
# -------------------------------
def add_triple_interactions(df):
    for color in ["red", "green", "blue"]:
        max_d = df[[f"d_{color}_mean_{i}_{i+1}" for i in range(1,5)]].max(axis=1)
        # Area × status_progress × max delta color
        df[f"area_x_status_progress_x_max_d_{color}"] = df["area"] * df["status_progress"] * max_d
    return df

# -------------------------------
# Appliquer toutes les interactions
# -------------------------------
def add_all_interactions(df):
    df = add_geometry_status_interactions(df)
    df = add_rgb_geometry_interactions(df)
    df = add_rgb_status_interactions(df)
    df = add_triple_interactions(df)
    return df

# -------------------------------
# Appliquer à train et test
# -------------------------------
train_gdf = add_all_interactions(train_gdf)
test_gdf  = add_all_interactions(test_gdf)


In [ ]:
le = LabelEncoder()
Y = le.fit_transform(train_gdf["change_type"])

# Ton mapping personnalisé
correction_map = {
    0: 3,  # 0  Commercial ->  3
    1: 0,  # 1  Demolition ->  0
    2: 4,  # 2  Industrial ->  4
    3: 5,  # 3  Mega Projects ->  5
    4: 2,  # 4  Residential ->  2
    5: 1   # 5  Road ->  1
}

# Appliquer le mapping personnalisé
Y = pd.Series(Y).map(correction_map).values

In [ ]:
# -------------------------
# 8) Nettoyage colonnes inutiles
# -------------------------
drop_cols = date_cols + status_cols + ["geometry", "change_type", "convex_hull_area"]
train_gdf_final = train_gdf.drop(columns=drop_cols).replace([np.inf, -np.inf], np.nan)
test_gdf_final  = test_gdf.drop(columns=drop_cols).replace([np.inf, -np.inf], np.nan)

In [ ]:
# -------------------------
# Vérification des dimensions
# -------------------------
print("Train dataset : ", train_gdf_final.shape, "->", train_gdf_final.shape[0], "lignes,", train_gdf_final.shape[1], "colonnes")
print("Test dataset  : ", test_gdf_final.shape, "->", test_gdf_final.shape[0], "lignes,", test_gdf_final.shape[1], "colonnes")

Train dataset :  (296146, 135) -> 296146 lignes, 135 colonnes
Test dataset  :  (120526, 135) -> 120526 lignes, 135 colonnes


In [ ]:
Y.shape

(296146,)

In [ ]:
test_gdf_final_1 = test_gdf_final.copy()
train_gdf_final_1 = train_gdf_final.copy()

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# On garde le dataset original pour comparaison
all_cols = train_gdf_final_1.columns

# Supprimer les colonnes avec variance faible
selector = VarianceThreshold(threshold=0.01)
selector.fit(train_gdf_final_1)  # sur train seulement

# Récupérer les colonnes qui restent
cols_kept = train_gdf_final_1.columns[selector.get_support()]
train_gdf_final_1 = train_gdf_final_1[cols_kept]
test_gdf_final_1  = test_gdf_final_1[cols_kept]

print("Après suppression des colonnes constantes :")
print("Train shape:", train_gdf_final_1.shape)
print("Test shape :", test_gdf_final_1.shape)

# Colonnes supprimées
cols_dropped = [col for col in all_cols if col not in cols_kept]

print(f"Nombre de colonnes supprimées : {len(cols_dropped)}")
print("Colonnes supprimées :", cols_dropped)


Après suppression des colonnes constantes :
Train shape: (296146, 128)
Test shape : (120526, 128)
Nombre de colonnes supprimées : 7
Colonnes supprimées : ['solidity', 'geography_type_Hills', 'geography_type_Snow', 'green_cv', 'blue_cv', 'green_ratio', 'red_ratio']


In [ ]:
# Matrice de corrélation
corr_matrix = train_gdf_final_1.corr().abs()

# Se concentrer sur la partie supérieure pour éviter les doublons
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))


In [ ]:
# Identifier les colonnes à supprimer si corrélation > 0.99
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]

# Supprimer ces colonnes
train_gdf_final_1 = train_gdf_final_1.drop(columns=to_drop)
test_gdf_final_1  = test_gdf_final_1.drop(columns=to_drop)

print(f"Colonnes supprimées pour forte corrélation : {len(to_drop)}", to_drop)

print("Train shape final :", train_gdf_final_1.shape)
print("Test shape final  :", test_gdf_final_1.shape)


Colonnes supprimées pour forte corrélation : 36 ['img_green_std_date1', 'img_blue_std_date1', 'img_blue_mean_date2', 'img_green_std_date2', 'img_blue_std_date2', 'img_green_std_date3', 'img_blue_std_date3', 'img_blue_mean_date4', 'img_green_std_date4', 'img_blue_std_date4', 'img_blue_mean_date5', 'img_green_std_date5', 'img_blue_std_date5', 'log_perimeter', 'elongation', 'construction_speed', 'first_status', 'last_status', 'status_range', 'green_mean_temporal', 'green_std_temporal', 'd_green_std_1_2', 'd_green_std_2_3', 'd_green_std_3_4', 'd_green_std_4_5', 'blue_mean_temporal', 'blue_std_temporal', 'd_blue_std_1_2', 'd_blue_std_2_3', 'd_blue_std_3_4', 'd_blue_mean_4_5', 'd_blue_std_4_5', 'blue_slope_x_area', 'blue_max_delta_x_status_progress', 'area_x_status_progress_x_max_d_green', 'area_x_status_progress_x_max_d_blue']
Train shape final : (296146, 92)
Test shape final  : (120526, 92)


In [ ]:
# ---------------------------------------------------------
# Sauvegarde des datasets finaux nettoyés en CSV
# ---------------------------------------------------------

# Sauvegarde du jeu d'entraînement
train_gdf_final_1.to_csv(DATA_PATH / "train_features_cleaned.csv", index=True)

# Sauvegarde du jeu de test
test_gdf_final_1.to_csv(DATA_PATH / "test_features_cleaned.csv", index=True)

print("Fichiers CSV sauvegardés avec succès !")

In [ ]:
import pandas as pd

# Crée un DataFrame pour Y
target_df = pd.DataFrame({
    "change_type": Y
}, index=train_gdf_final_1.index)  # garde le même index que tes features

# Sauvegarde en CSV
target_df.to_csv(DATA_PATH / "train_target_corr.csv", index=True)  # index=True pour garder l'index
print("Fichier 'train_target_corr.csv' généré avec succès !")